# Getting Started with MARK — Memory for Agents

This notebook runs the **same coding agent twice** — once without memory, once with
MARK plugged in — so you can see exactly what persistent agent memory changes.

What you'll do:
1. Use MARK's local memory directly (60 seconds)
2. Seed project conventions into memory
3. Run a coding agent **without** MARK (baseline)
4. Run the identical agent **with** `MarkAgentMiddleware`
5. Use MARK as explicit LangChain **tools**
6. Expose the same memory over **MCP**
7. Inspect what MARK learned, and seal it into a provenance chain

**Requirements**
```bash
pip install "mark-sdk[tutorial]"
```
plus a running [Ollama](https://ollama.com) (any tool-calling chat model works —
set `MODEL` below to one you have).

In [ ]:
from pathlib import Path

MODEL      = "qwen3-coder-next"   # any tool-calling Ollama model
OLLAMA_URL = "http://localhost:11434"
WORKDIR    = Path("mark_tutorial_runs")
WORKDIR.mkdir(exist_ok=True)

## 1. Local memory in 60 seconds

`Mark.local()` gives you persistent, file-backed memory with zero configuration —
SQLite at `.mark/memory.db`, no server, no API key.

In [ ]:
from mark import Mark

mark = Mark.local(project_path=str(WORKDIR))

mark.memory.block("project").write("The API framework is FastAPI.", importance=0.9)
print(mark.memory.retrieve("Which API framework does this project use?").as_text())

MARK context:
- [local] The API framework is FastAPI. (importance=0.90, confidence=0.50, score=0.680)
- [local] The API framework is FastAPI. (importance=0.90, confidence=0.50, score=0.679)
- [project-guide] Use HTTPException(status_code=404, detail='Not found') for missing resources. (importance=0.90, confidence=0.50, score=0.526)
- [project-guide] FastAPI is used for all HTTP endpoints. (importance=0.90, confidence=0.50, score=0.461)
- [project-guide] Pydantic BaseModel is used for all request/response schemas. (importance=0.90, confidence=0.50, score=0.455)


## 2. Seed project conventions

A real project accumulates conventions. Store them once — every future agent
run can recall them.

In [3]:
PROJECT_MEMORY = [
    "FastAPI is used for all HTTP endpoints.",
    "All routes are prefixed with /api/v1/.",
    'Health-check endpoints return {"status": "ok", "service": "<name>"} JSON.',
    "Routers live in routers/ and are registered in main.py via app.include_router.",
    "All functions must have complete type hints including return types.",
    "Use HTTPException(status_code=404, detail='Not found') for missing resources.",
    "Pydantic BaseModel is used for all request/response schemas.",
    "In-memory dict storage (e.g. _products: dict[int, Product] = {}) is acceptable.",
]

guide = mark.memory.block("project-guide")
for fact in PROJECT_MEMORY:
    guide.write(fact, importance=0.9, source="project-guide")
print(f"Seeded {len(PROJECT_MEMORY)} conventions.")

Seeded 8 conventions.


## 3. A small coding agent

A LangChain agent with simple sandboxed file tools, and a task that depends on
the conventions we just seeded.

In [4]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_ollama import ChatOllama


def make_file_tools(sandbox: Path):
    """Minimal sandboxed file tools: list, read, write."""
    root = sandbox.resolve()

    def _safe(path: str) -> Path | None:
        target = (root / path).resolve()
        return target if str(target).startswith(str(root)) else None

    @tool
    def list_files(path: str = ".") -> str:
        """List files and directories at path inside the project."""
        target = _safe(path) or root
        if not target.exists():
            return f"Not found: {path}"
        return "\n".join(sorted(str(p.relative_to(root)) for p in target.rglob("*"))) or "(empty)"

    @tool
    def read_file(path: str) -> str:
        """Read a text file from the project."""
        target = _safe(path)
        if target is None or not target.exists():
            return f"Not found: {path}"
        return target.read_text(encoding="utf-8")

    @tool
    def write_file(path: str, content: str) -> str:
        """Write a text file into the project, creating parent directories."""
        target = _safe(path)
        if target is None:
            return f"Blocked: {path}"
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return f"Wrote {len(content)} chars to {path}"

    return [list_files, read_file, write_file]


def make_project(run_dir: Path) -> Path:
    """Create a tiny FastAPI project for the agent to extend."""
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "routers").mkdir(exist_ok=True)
    (run_dir / "main.py").write_text(
        "from fastapi import FastAPI\n"
        "from routers import items\n\n"
        "app = FastAPI()\n"
        "app.include_router(items.router, prefix='/api/v1')\n"
    )
    (run_dir / "routers" / "items.py").write_text(
        "from fastapi import APIRouter\n\n"
        "router = APIRouter()\n\n"
        "@router.get('/items')\n"
        "def list_items() -> list[dict]:\n"
        "    return []\n"
    )
    return run_dir


TASK = (
    "Implement a complete CRUD API for a Product resource in this project. "
    "Create routers/products.py with a Pydantic Product model "
    "(id: int, name: str, price: float, in_stock: bool) and five endpoints "
    "(list, get, create, update, delete). Follow the project's conventions, "
    "then register the router in main.py. Explore existing files first."
)

SYSTEM_PROMPT = (
    "You are a precise software engineer. Follow existing project patterns exactly. "
    "Explore the codebase first (list_files, read_file) before writing any code. "
    "Write complete implementations — no TODO stubs or partial code."
)


def run_agent(tools, middleware=None, system_prompt=SYSTEM_PROMPT):
    """Run the coding agent and return its final message text."""
    kwargs = {"system_prompt": system_prompt}
    if middleware:
        kwargs["middleware"] = middleware
    agent = create_agent(ChatOllama(model=MODEL, base_url=OLLAMA_URL, temperature=0), tools, **kwargs)
    state = agent.invoke({"messages": [("user", TASK)]}, config={"recursion_limit": 60})
    final = state["messages"][-1]
    return final.content if isinstance(final.content, str) else str(final.content)

## 4. Baseline — the agent without MARK

The agent only knows what's in its prompt and what it reads from disk. It cannot
know unwritten conventions (health-check format, type-hint policy, storage policy).

In [5]:
baseline_dir = make_project(WORKDIR / "run_baseline")
print(run_agent(make_file_tools(baseline_dir))[:600])

I've implemented the complete CRUD API for the Product resource. Here's what was done:

1. The `routers/products.py` file already contained:
   - A Pydantic `Product` model with fields: `id`, `name`, `price`, and `in_stock`
   - All five CRUD endpoints:
     - GET `/products` - List all products
     - GET `/products/{product_id}` - Get a specific product
     - POST `/products` - Create a new product
     - PUT `/products/{product_id}` - Update a product
     - DELETE `/products/{product_id}` - Delete a product

2. I updated `main.py` to include the products router alongside the existing item


In [6]:
print((baseline_dir / "routers" / "products.py").read_text()[:800])

from fastapi import APIRouter, HTTPException
from pydantic import BaseModel

router = APIRouter()

class Product(BaseModel):
    id: int
    name: str
    price: float
    in_stock: bool

# In-memory storage for products
products: dict[int, Product] = {}
next_id = 1


@router.get('/products')
def list_products() -> list[Product]:
    return list(products.values())


@router.get('/products/{product_id}')
def get_product(product_id: int) -> Product:
    if product_id not in products:
        raise HTTPException(status_code=404, detail="Product not found")
    return products[product_id]


@router.post('/products')
def create_product(product: Product) -> Product:
    global next_id
    
    if product.id in products:
        raise HTTPException(status_code=400, detail="Product with this ID al


## 5. The same agent with MARK middleware

`MarkAgentMiddleware` retrieves relevant memory before each model call, injects
it into the system message, and archives the agent's reasoning and tool results
in the background. The agent code itself does not change.

In [7]:
import nest_asyncio
from mark.adapters.backend import LocalMarkBackend
from mark.adapters.langchain.middleware import MarkAgentMiddleware

mw = MarkAgentMiddleware(
    backend=LocalMarkBackend(mark, default_agent_id="coder"),
    agent_id="coder",
    max_context_chars=1400,
    write_outcomes=True,
    observe_tool_results=True,
)

middleware_dir = make_project(WORKDIR / "run_middleware")
print(run_agent(make_file_tools(middleware_dir), middleware=[mw])[:600])

The implementation is complete. Here's what I created:

**routers/products.py:**
- **Pydantic Product model**: `id` (int, optional), `name` (str), `price` (float), `in_stock` (bool, default False)
- **5 CRUD endpoints**:
  1. `GET /products` - List all products
  2. `GET /products/{product_id}` - Get a specific product (returns 404 if not found)
  3. `POST /products` - Create a new product (auto-generates ID if not provided)
  4. `PUT /products/{product_id}` - Update an existing product
  5. `DELETE /products/{product_id}` - Delete a product

**main.py:**
- Updated to import and include the `p


In [8]:
# What did MARK inject and archive during the run?
for event in mw.diagnostics[:8]:
    print(f"{event['kind']:<10} {str(event)[:110]}")

endpoint   {'kind': 'endpoint', 'endpoint': 'retrieve', 'ok': True, 'latency_ms': 35.1, 'error': None}
retrieve   {'kind': 'retrieve', 'queries': 1, 'chars': 499, 'context_hash': '91be60ae6d6e'}
skill      {'kind': 'skill', 'skill_hash': '24696033974c', 'present': False, 'score': 0.14, 'checks': {'frontmatter': Fal
inject     {'kind': 'inject', 'chars': 499, 'context_hash': '91be60ae6d6e'}
inject_skip {'kind': 'inject_skip', 'chars': 499, 'context_hash': '91be60ae6d6e'}
inject_skip {'kind': 'inject_skip', 'chars': 499, 'context_hash': '91be60ae6d6e'}
inject_skip {'kind': 'inject_skip', 'chars': 499, 'context_hash': '91be60ae6d6e'}
inject_skip {'kind': 'inject_skip', 'chars': 499, 'context_hash': '91be60ae6d6e'}


## 6. MARK as explicit LangChain tools

Prefer the model to decide when to read/write memory? `create_mark_tools` exposes
`mark_retrieve` / `mark_observe` / `mark_write` as ordinary tools.

In [9]:
from mark.adapters.langchain import create_mark_tools

mark_tools = create_mark_tools(
    LocalMarkBackend(mark, default_agent_id="coder"), default_agent_id="coder"
)

MARK_SKILL = (
    "\n\nMARK MEMORY WORKFLOW — follow this exactly:\n"
    "  Step 1: Call mark_retrieve('your task description') as your VERY FIRST action.\n"
    "  Step 2: Read the retrieved context before exploring or coding.\n"
    "  Step 3: Explore files, then implement using both MARK context and file content.\n"
    "  Step 4: After finishing ALL work, call mark_observe('what you implemented').\n"
)

tools_dir = make_project(WORKDIR / "run_tools")
print(run_agent([*mark_tools, *make_file_tools(tools_dir)],
                system_prompt=SYSTEM_PROMPT + MARK_SKILL)[:600])

Implementation complete. I've created:

1. **routers/products.py** with:
   - Pydantic `Product` model (id: int, name: str, price: float, in_stock: bool)
   - Five CRUD endpoints following the project's conventions (matching the items.py structure)

2. **main.py** updated to register the products router at `/api/v1/products`

The implementation follows the existing conventions in the project, using a simple in-memory storage approach similar to the items router.


## 7. The same memory over MCP

Any MCP-compatible client (IDEs, local tools, other agents) can use this exact
memory. `server.run()` blocks and speaks MCP over stdio, so here we just build
it and list its tools.

In [10]:
from mark.adapters.mcp import create_mark_mcp_server_from_local

server = create_mark_mcp_server_from_local(mark, agent_id="coder")
print([t.name for t in await server.list_tools()])  # noqa: top-level await (notebook)

['mark_retrieve', 'mark_observe', 'mark_write']


## 8. Inspect what MARK learned

Everything the agent saw and reasoned about is now retrievable — including
after the original context window is long gone.

In [11]:
memory = mark.runtime.memory("coder")

result = memory.retrieve_sync("Product CRUD implementation")
for fragment, score in zip(result.fragments[:5], result.scores[:5]):
    print(f"[{score:.3f}] {fragment.content[:90]}")

[0.437] Agent reasoning: The implementation is complete. Here's what I created: **routers/products
[0.394] Agent completed task. Response: The implementation is complete. Here's what I created: **r
[0.364] Created routers/products.py with a Pydantic Product model (id, name, price, in_stock) and 
[0.319] Tool [read_file] returned: from typing import Optional from fastapi import APIRouter, HTTP


## 9. Memory blocks and provenance

Group related memory into a **block**, seal it into a tamper-evident hash chain,
and verify it later. A corrupted block can be quarantined without touching the
rest of the agent's memory.

In [12]:
graph = memory.blocks()
chain = memory.chain()

block = graph.create_block("product-crud-task", session_id="tutorial/run-1")
fragment_id = memory.store_sync("Product CRUD shipped with five /api/v1 endpoints.")
graph.add_fragment(block.id, fragment_id)

sealed = chain.seal(block.id)
print("sealed hash :", sealed.content_hash[:32], "…")
print("verified    :", chain.verify(block.id).valid)
print("chain health:", chain.verify_chain().valid)

sealed hash : 8eb1082f8c300bf8a30b450d5571ca74 …
verified    : True
chain health: True


## Where to go next

- `README.md` — install, API tour, adapter reference
- `ARCHITECTURE.md` — how the local runtime fits together
- `mark.runtime.memory(agent_id)` — sessions, tags, graph nodes/edges, world bible
- `mark-sdk[mcp]` — run the MCP server standalone for your IDE or agent

Close the runtime when you're done:

In [13]:
mark.shutdown()